# 07. Auditoría de Tags Editoriales y Propuesta de Normalización

Analizamos las etiquetas reales (`post_tag`) de los articulos de Anfibia, extraídas de la base de datos original.
Comparamos las etiquetas  frente a los **tópicos semánticos latentes** (BERTopic) para diagnosticar problemas de cobertura y solapamiento.

In [3]:
# 1. Parche de compatibilidad obligatorio (Numba/Coverage)
import coverage.types as ct
for name, fallback in {
    'Tracer': getattr(ct, 'TTracer', object),
    'TShouldTraceFn': object,
    'TShouldStartContextFn': object,
    'TFileDisposition': object,
}.items():
    if not hasattr(ct, name):
        setattr(ct, name, fallback)

# 2. Imports generales
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
import difflib
import unicodedata
import math

warnings.filterwarnings("ignore")

# 3. Rutas
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

from bertopic.backend._utils import select_backend

INPUT_DIR = PROJECT_ROOT / "outputs" / "bertopic_acotado"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "analisis_etiquetas"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# 4. Cargar el Parquet ya depurado (ya contiene ID, doc_id, topic final, etc.)
df = pd.read_parquet(INPUT_DIR / "df_completo.parquet")
if "doc_id" not in df.columns:
    df["doc_id"] = df.index

# Asegurar post_year si no quedó guardado explícitamente
if "post_year" not in df.columns:
    df["post_year"] = pd.to_datetime(df["post_date"].astype(str).str.split(" ").str[0], errors="coerce").dt.year

# 5. Cargar el modelo final con 80 keywords y asociar transformer
topic_model = BERTopic.load(str(INPUT_DIR / "modelo_completo"))
raw_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
topic_model.embedding_model = select_backend(raw_model)

topic_info = topic_model.get_topic_info().set_index("Topic")["Name"].to_dict()

print(f"Total de notas cruzadas: {len(df):,}")

2026-10-15 22:48:11,285 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6795.85it/s]


Total de notas cruzadas: 3,556


## 1. Cobertura Histórica de Tags

In [4]:
df.columns

Index(['doc_id', 'source_url', 'title_text', 'tagline', 'autor_names',
       'post_date', 'post_date_dt', 'category', 'post_tag', 'topic',
       'document_text_final', 'post_year'],
      dtype='str')

In [5]:
total = len(df)
df["tiene_tag"] = df["post_tag"].notna() & (df["post_tag"].str.strip() != "")

con_tag = df["tiene_tag"].sum()
sin_tag = total - con_tag

print(f"Notas con etiquetas: {con_tag:,} ({(con_tag/total)*100:.1f}%)")
print(f"Notas SIN etiquetas: {sin_tag:,} ({(sin_tag/total)*100:.1f}%)")

fig = go.Figure(data=[go.Pie(labels=["Con Tags", "Sin Tags"], values=[con_tag, sin_tag], hole=.4)])
fig.update_layout(title="Cantidad de etiquetas en los articulos", width=500, height=400)
fig.show()

Notas con etiquetas: 1,589 (44.7%)
Notas SIN etiquetas: 1,967 (55.3%)


In [6]:
# Agrupamos por año: Total de notas vs Notas con tag
evolucion = (
    df.groupby("post_year")
    .agg(
        total_notas=("doc_id", "count"),
        notas_con_tag=("tiene_tag", "sum")
    )
    .reset_index()
)
evolucion["notas_sin_tag"] = evolucion["total_notas"] - evolucion["notas_con_tag"]
evolucion["pct_con_tag"] = (evolucion["notas_con_tag"] / evolucion["total_notas"] * 100).round(1)
evolucion["pct_sin_tag"] = (evolucion["notas_sin_tag"] / evolucion["total_notas"] * 100).round(1)

# Filtramos posibles años nulos o erróneos
#evolucion = evolucion[evolucion["post_year"] > 2010]


In [7]:
#Gráfico de barras apiladas
fig_evo = go.Figure()

# Texto condicional: oculta el porcentaje si la barra es menor al 6% para evitar desbordes
textos_verdes = [f"{p}%" if p > 6 else "" for p in evolucion["pct_con_tag"]]

# Barra inferior: Notas que SÍ tienen tag
fig_evo.add_trace(go.Bar(
    x=evolucion["post_year"],
    y=evolucion["notas_con_tag"],
    name="Con Etiquetas",
    marker_color='#2ca02c',
    text=textos_verdes,
    textposition='inside',
    textangle=0,  # <--- ACÁ ESTÁ EL CAMBIO: 0 para texto horizontal
    insidetextanchor='middle',
    customdata=evolucion["pct_con_tag"],
    hovertemplate="<b>%{x}</b><br>Con etiquetas: %{y} notas (%{customdata}%)<extra></extra>"
))

# Barra superior: Notas SIN tag
fig_evo.add_trace(go.Bar(
    x=evolucion["post_year"],
    y=evolucion["notas_sin_tag"],
    name="Sin Etiquetas",
    marker_color='#d62728',
    customdata=evolucion["pct_sin_tag"],
    hovertemplate="<b>%{x}</b><br>Sin etiquetas: %{y} notas (%{customdata}%)<extra></extra>"
))

fig_evo.update_layout(
    barmode='stack',
    title="<b>Articulos etiquetados por Año</b>",
    xaxis_title="Año de Publicación",
    yaxis_title="Cantidad de Artículos",
    height=500,
    width=900,
    plot_bgcolor='rgba(0,0,0,0)',
    legend=dict(orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.5)
)
fig_evo.update_xaxes(type='category')
fig_evo.write_html(OUTPUT_DIR / "03_evolucion_etiquetado.html")
fig_evo.show()

## 2. Fragmentación y Uso Único de Etiquetas
Ocurre que muchos artiuculos tienen mas de una etiqueta.
Separamos las etiquetas múltiples (divididas por `;`) y analizamos su frecuencia.

In [8]:
TOP = 15
# Desanidar tags separados por punto y coma
tags_list = (
    df["post_tag"]
    .dropna()
    .str.split(r";")
    .explode()
    .str.strip()
    .loc[lambda x: x != ""]
)

tag_counts = tags_list.value_counts()
tags_unicos = len(tag_counts)
tags_una_vez = (tag_counts == 1).sum()

print(f"Cantidad de etiquetas únicas creadas: {tags_unicos:,}")
print(f"Etiquetas usadas en 1 sola nota: {tags_una_vez:,} ({(tags_una_vez/tags_unicos)*100:.1f}%)")

print(f"\nTop {TOP} Etiquetas más usadas:")
display(tag_counts.head(TOP).to_frame("Frecuencia"))

Cantidad de etiquetas únicas creadas: 881
Etiquetas usadas en 1 sola nota: 444 (50.4%)

Top 15 Etiquetas más usadas:


,Frecuencia
post_tag,
Internacional,238
Javier Milei,162
Derechas,114
Zapping cultural,84
Derechos Humanos,77
UNSAM,67
Chile,65
Feminismos,62
Peronismo,61


Analisemos esas etiquetas que solo se usaron una vez.

In [9]:
# Muestreo y Diagnóstico de Tags Unicos

unicos = tag_counts[tag_counts == 1].index.to_series().reset_index(drop=True)

df_diag = pd.DataFrame({"tag": unicos})
df_diag["cant_palabras"] = df_diag["tag"].str.split().str.len()

# 3. Distribución por cantidad de palabras
print("Distribución de longitud (palabras por tag unico):")
for n_words, total in df_diag["cant_palabras"].value_counts().sort_index().items():
    pct = (total / len(df_diag)) * 100
    print(f" • {n_words} palabra(s): {total} tags ({pct:.1f}%)")
    
print("\n" + "=" * 60)
print(f"Muestra aleatoria de 30 tags Unicos (de un total de {len(df_diag)}):")
print("=" * 60)

# 4. Muestra representativa
muestra = df_diag["tag"].sample(n=min(30, len(df_diag)), random_state=42).tolist()
for i, t in enumerate(muestra, 1):
    print(f"{i:2d}. {t}")

Distribución de longitud (palabras por tag unico):
 • 1 palabra(s): 232 tags (52.3%)
 • 2 palabra(s): 159 tags (35.8%)
 • 3 palabra(s): 39 tags (8.8%)
 • 4 palabra(s): 11 tags (2.5%)
 • 5 palabra(s): 3 tags (0.7%)

Muestra aleatoria de 30 tags Unicos (de un total de 444):
 1. Nora Dalmasso
 2. cancelación
 3. Honduras
 4. Canal de Panamá
 5. Xiomara Castro
 6. Autista
 7. Familiares
 8. Noche
 9. Pep
10. Imaginación política
11. Ciudades
12. Amistad
13. #EAAF
14. Inteligencia
15. Ecología
16. gendarmería
17. Derechos sociales
18. Disidencias sexuales
19. Moda
20. Beca crónica
21. Randazzo
22. Animales
23. Palito Ortega
24. Liliana Herrero
25. Instagram
26. Vínculos
27. Invasión
28. Ríos entubados
29. Fundamentalistas del Aire Acondicionado
30. Soledad Acuña


## 3. Etiquetas en las categorias

Recordamos como se distribuyen las categorias.

In [13]:


# 1. Normalizar y explotar manteniendo el dataframe listo
df_cat_exploded = (
    df[["doc_id", "post_year", "category", "post_tag"]]
    .copy()
    .assign(
        categoria_normalizada=lambda d: d["category"]
        .fillna("Sin categoria")
        .astype(str)
        .apply(lambda x: [p.strip() for p in re.split(r"[;,]", x) if p.strip()] or ["Sin categoria"])
    )
    .explode("categoria_normalizada")
    .reset_index(drop=True)
)

# 2. Conteo de artículos por categoría
df_cat_dist = (
    df_cat_exploded.groupby("categoria_normalizada")["doc_id"]
    .nunique()
    .reset_index(name="notas")
    .sort_values("notas", ascending=True)
)

# 3. Gráfico 
fig_cat = px.bar(
    df_cat_dist, x="notas", y="categoria_normalizada", orientation="h",
    title="<b>Distribución de Articulos por Categoría</b>",
    text="notas", color="notas", color_continuous_scale="Blues"
)
fig_cat.update_traces(textposition="outside")
fig_cat.update_layout(
    height=400, plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="#fff",
    coloraxis_showscale=False, xaxis_title="", yaxis_title="",
    margin=dict(l=150, r=40, t=50, b=30)
)
fig_cat.write_html(OUTPUT_DIR / "03a_distribucion_categorias.html")
fig_cat.show()

In [16]:
# ==============================================================================
# DISTRIBUCIÓN DE ETIQUETAS ÚNICAS POR CATEGORÍA
# ==============================================================================
df_cat_tag_exploded = (
    df[["doc_id", "category", "post_tag"]]
    .dropna(subset=["post_tag"])
    .copy()
    .assign(
        categoria_normalizada=lambda d: d["category"]
        .fillna("Sin categoria")
        .astype(str)
        .apply(lambda x: [p.strip() for p in re.split(r"[;,]", x) if p.strip()] or ["Sin categoria"]),
        tag=lambda d: d["post_tag"].str.split(r"[;,]")
    )
    .explode("categoria_normalizada")
    .explode("tag")
    .assign(tag=lambda d: d["tag"].str.strip().str.title())
    .loc[lambda d: (d["tag"] != "") & (d["tag"] != "Nan")]
    .reset_index(drop=True)
)

df_tags_dist_cat = (
    df_cat_tag_exploded.groupby("categoria_normalizada")["tag"]
    .nunique()
    .reset_index(name="etiquetas")
    .sort_values("etiquetas", ascending=True)
)

fig_tags_cat = px.bar(
    df_tags_dist_cat, 
    x="etiquetas", 
    y="categoria_normalizada", 
    orientation="h",
    title="<b>Distribución de Etiquetas Únicas por Categoría</b>",
    text="etiquetas", 
    color="etiquetas", 
    color_continuous_scale="Blues"
)
fig_tags_cat.update_traces(textposition="outside")
fig_tags_cat.update_layout(
    height=400, 
    plot_bgcolor="rgba(0,0,0,0)", 
    paper_bgcolor="#fff",
    coloraxis_showscale=False, 
    xaxis_title="", 
    yaxis_title="",
    margin=dict(l=150, r=40, t=50, b=30)
)
fig_tags_cat.write_html(OUTPUT_DIR / "03d_distribucion_etiquetas_por_categoria.html")
fig_tags_cat.show()

Mostramos como se distribuyen N-TOP etiquetas sobre las categorias.

In [19]:
TOP = 15
# 1. Columnas necesarias y descarte de nulos
df_cruce = df_cat_exploded.dropna(subset=["post_tag"])[["doc_id", "post_year", "categoria_normalizada", "post_tag"]].copy()

# 2. Desanidar tags y normalizar
df_cruce["tag"] = df_cruce["post_tag"].str.split(r"[;,]")
df_cruce = df_cruce.explode("tag")
df_cruce["tag"] = df_cruce["tag"].str.strip().str.title()
df_cruce = df_cruce[df_cruce["tag"] != ""]

# 3. Eliminar duplicados nota-tag-categoria y RESETEAR INDICE
df_cruce = df_cruce.drop_duplicates(subset=["doc_id", "tag", "categoria_normalizada"]).reset_index(drop=True)

# 4. Top N por notas únicas
top_tags = df_cruce.groupby("tag")["doc_id"].nunique().nlargest(TOP)
mapeo_labels = {t: f"{t} ({n})" for t, n in top_tags.items()}

# 5. Filtrar el top y resetear índice de nuevo para evitar desalineaciones en crosstab
df_top = df_cruce[df_cruce["tag"].isin(top_tags.index)].copy().reset_index(drop=True)
df_top["tag_label"] = df_top["tag"].map(mapeo_labels)

# 6. Matriz de contingencia ordenada
orden_filas = [mapeo_labels[t] for t in top_tags.index][::-1]
matriz = pd.crosstab(df_top["tag_label"], df_top["categoria_normalizada"]).reindex(orden_filas)
matriz = matriz.loc[:, matriz.sum(axis=0) > 0]

In [20]:
CATEGORIAS_MOSTRAR = ['Ambiente', 'Amor y sexo', 'Educación', 'Otras', 
                      #'Culturas',
                      #'Política y Economía', 
                      # 'Políticas de géneros', 
                      'Salud', 'Tecnologías'
                      ]

matriz_mostrar = matriz[CATEGORIAS_MOSTRAR]

# 7. Render del Heatmap
fig_heatmap = px.imshow(
    matriz_mostrar,
    labels=dict(x="Categoría", y="", color="Notas"),
    x=matriz_mostrar.columns,
    y=matriz_mostrar.index,
    color_continuous_scale="Blues",
    aspect="auto",
    text_auto=True
)

fig_heatmap.update_traces(
    hovertemplate="<b>Tag:</b> %{y}<br><b>Categoría:</b> %{x}<br><b>Notas en sección:</b> %{z}<extra></extra>"
)

fig_heatmap.update_layout(
    title=f"<b>Top {TOP} Etiquetas (Total) vs Categorías</b>",
    xaxis_title="",
    yaxis_title="",
    height=max(500, TOP * 32),
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="#fff",
    margin=dict(l=180, b=100)
)
fig_heatmap.update_xaxes(tickangle=-45)

fig_heatmap.write_html(OUTPUT_DIR / f"03b_top{TOP}_tags_heatmap.html")
fig_heatmap.show()

Veamos como se distribuyen las etiquetas por categoria teniendo en cuenta el año.

In [21]:
#Donas por Categoría con Dropdown 


TOP_TAGS = 7

CATEGORIAS = [c for c in df_cruce["categoria_normalizada"].unique()]
AÑOS = ["Global"] + sorted([str(int(a)) for a in df_cruce["post_year"].dropna().unique()])

n_cols = 3
n_rows = math.ceil(len(CATEGORIAS) / n_cols)

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    specs=[[{"type": "domain"}] * n_cols for _ in range(n_rows)],
    subplot_titles=[f"<b>{c}</b>" for c in CATEGORIAS],
    vertical_spacing=0.18, horizontal_spacing=0.12
)

# Paleta limpia y formateo de texto exterior
paleta_colores = px.colors.qualitative.Prism

def wrap_label(txt, max_len=14):
    if len(txt) > max_len and " " in txt:
        partes = txt.split(" ")
        medio = len(partes) // 2
        return " ".join(partes[:medio]) + "<br>" + " ".join(partes[medio:])
    return txt

# 1. Pre-cálculo de frecuencias vectorizado
conteo_años = df_cruce.groupby(["post_year", "categoria_normalizada", "tag"]).size()
conteo_global = df_cruce.groupby(["categoria_normalizada", "tag"]).size()

# 2. Inyección de trazas
for i_periodo, periodo in enumerate(AÑOS):
    for idx, cat in enumerate(CATEGORIAS):
        r, c = (idx // n_cols) + 1, (idx % n_cols) + 1
        
        serie = conteo_global.get(cat) if periodo == "Global" else conteo_años.get((int(periodo), cat))
        top = serie.nlargest(TOP_TAGS) if serie is not None else pd.Series(dtype=int)

        if not top.empty:
            labels_raw = top.index.tolist()
            labels_plot = [wrap_label(l) for l in labels_raw]
            values = top.values.tolist()
            colores = paleta_colores[:len(labels_plot)]
            textinfo = "label+percent"
        else:
            labels_raw, labels_plot, values = ["Sin datos"], ["Sin datos"], [1]
            colores = ["#eaeaea"]
            textinfo = "none"

        fig.add_trace(
            go.Pie(
                labels=labels_plot,
                customdata=labels_raw,
                values=values,
                hole=0.60,
                marker=dict(colors=colores, line=dict(color="#ffffff", width=2)),
                textposition="outside",
                textinfo=textinfo,
                textfont=dict(size=10),
                hovertemplate="<b>%{customdata}</b><br>Notas: %{value} (%{percent})<extra></extra>",
                showlegend=False,
                visible=(i_periodo == 0)
            ),
            row=r, col=c
        )

# 3. Dropdown funcional y compacto
total_trazas, k = len(fig.data), len(CATEGORIAS)
botones = [
    dict(
        method="update",
        label=f"  {p}  ",
        args=[
            {"visible": [i // k == step for i in range(total_trazas)]},
            {"title": f"<b>Distribución de Tags por Categoría — Período: {p}</b>"}
        ]
    )
    for step, p in enumerate(AÑOS)
]

fig.update_layout(
    updatemenus=[dict(
        buttons=botones,
        direction="down",
        x=0.0, y=1.14,
        xanchor="left", yanchor="top",
        bgcolor="#ffffff", bordercolor="#d1d5db",
        font=dict(size=11, color="#1f2937"),
        showactive=True
    )],
    title_text="<b>Etiquetas por categoria</b>",
    height=420 * n_rows + 40,
    width=1500,
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="#ffffff",
    margin=dict(t=120, b=40, l=40, r=40)
)

fig.write_html(OUTPUT_DIR / "03c_tags_donas_dropdown_temporal.html")
fig.show()

In [23]:
# Métrica rápida de ciclo de vida
vida_tags = (
    df_cruce.groupby("tag")
    .agg(
        primer_año=("post_year", "min"),
        ultimo_año=("post_year", "max"),
        total_notas=("doc_id", "nunique")
    )
    .assign(longevidad=lambda d: d["ultimo_año"] - d["primer_año"] + 1)
    .sort_values("total_notas", ascending=False)
)

vida_tags

,primer_año,ultimo_año,total_notas,longevidad
tag,,,,
Internacional,2012,2026,238,15
Javier Milei,2021,2026,162,6
Derechas,2018,2026,114,9
Zapping Cultural,2019,2026,84,8
Derechos Humanos,2014,2026,77,13
...,...,...,...,...
Urbanismo,2026,2026,1,1
Unión Europea,2026,2026,1,1
Universidad De Texas,2025,2025,1,1


## 3. Solapamiento: Tags vs Tópicos BERTopic
Vemos cómo un mismo tópico semántico fue etiquetado con múltiples tags manuales distintos.

In [24]:
# 1. Mapeo directo y asignación de nombre para el 100
df["topic_name"] = df["topic"].map(topic_info)
df.loc[df["topic"] == 100, "topic_name"] = "100_outliers_miscelaneo"

# 2. Desanidar tags directamente sobre df con la misma regex y formato
df_desanidado = (
    df.dropna(subset=["post_tag"])
    .copy()
    .assign(tag=lambda d: d["post_tag"].str.split(r"[;,]"))
    .explode("tag")
    .assign(tag=lambda d: d["tag"].str.strip().str.title())
    .loc[lambda d: (d["tag"] != "") & (d["tag"] != "Nan")]
    .reset_index(drop=True)
)

# 3. Agrupación y solapamiento (todos los tópicos, incluido el 100)
solapamiento = (
    df_desanidado
    .groupby(["topic", "topic_name"])
    .agg(
        notas_con_tag=("doc_id", "nunique"),
        tags_unicos_usados=("tag", "nunique"),
        ejemplos_tags=("tag", lambda x: " | ".join(pd.Series(x.unique())[:5]))
    )
    .reset_index()
    .sort_values("notas_con_tag", ascending=False)
)

display(solapamiento.head(10))
solapamiento.to_csv(OUTPUT_DIR / "auditoria_tags_por_topico.csv", index=False)

,topic,topic_name,notas_con_tag,tags_unicos_usados,ejemplos_tags
0,0,0_candidato_electoral_campaña_debate,335,257,Internacional | Venezuela | Brasil | Fútbol | ...
22,100,100_outliers_miscelaneo,222,317,Violencia Policial | Rossana Reguillo | Feria ...
1,1,1_literario_autor_escritor_poeta,143,177,Ba | Feria Del Libro | Encuentro Anfibio | Jua...
2,2,2_feminismo_feminista_aborto_abortar,131,125,Memoria Trans | Trabajo | #Niunamenos | Día In...
3,3,3_muerto_dictadura_muerte_morir,122,156,24M: Día De La Memoria Por La Verdad Y La Just...
5,5,5_déficit_economía_pobreza_económico,91,91,Fmi | Fondos Buitre | Clase Obrera | Trabajo |...
4,4,4_ambiental_ambiente_crisis_proyecto,86,130,Periodismo Narrativo | Internacional | Venezue...
6,6,6_futbolista_fifa_fútbol_brasil,55,56,Perfil Político | Barras Bravas | Boca Juniors...
7,7,7_pedagógico_enseñanza_escolar_educativo,50,68,Encuentro Anfibio | Feria Del Libro | Día Del ...
8,8,8_epidemia_pandemia_vacunar_virus,50,46,Vih | Rossana Reguillo | Rosariocronica | Rosa...


In [25]:
TOP_TAGS = 30
TOP_TOPICOS = 10


# 1. Filtrar tópicos válidos directamente
df_tags_topics = df_desanidado.copy()

# 2. Rankings Top N sin pisar la variable global tag_counts
top_tags_series = df_tags_topics.groupby("tag")["doc_id"].nunique().nlargest(TOP_TAGS)
top_nombres_tags = top_tags_series.index.tolist()
mapeo_etiqueta_total = {tag: f"{tag} ({conteo})" for tag, conteo in top_tags_series.items()}
orden_filas = [mapeo_etiqueta_total[tag] for tag in top_nombres_tags][::-1]

top_topics_series = df_tags_topics.groupby("topic_name")["doc_id"].nunique().nlargest(TOP_TOPICOS)
top_nombres_topics = top_topics_series.index.tolist()

# 3. Filtrar y mapear etiqueta
df_top = df_tags_topics[
    df_tags_topics["tag"].isin(top_nombres_tags) & 
    df_tags_topics["topic_name"].isin(top_nombres_topics)
].copy()
df_top["tag_label"] = df_top["tag"].map(mapeo_etiqueta_total)

# 4. Matriz de contingencia reordenada
matriz = (
    pd.crosstab(df_top["tag_label"], df_top["topic_name"])
    .reindex(index=orden_filas, columns=top_nombres_topics, fill_value=0)
)
matriz = matriz.loc[:, matriz.sum(axis=0) > 0]

# 5. Render del Heatmap
fig_heatmap = px.imshow(
    matriz,
    labels=dict(x="Tópico Semántico (BERTopic)", y="", color="Notas"),
    x=matriz.columns,
    y=matriz.index,
    color_continuous_scale="Blues",
    aspect="auto",
    text_auto=True
)

fig_heatmap.update_traces(
    hovertemplate="<b>Tag:</b> %{y}<br><b>Tópico:</b> %{x}<br><b>Notas solapadas:</b> %{z}<extra></extra>"
)

fig_heatmap.update_layout(
    title=f"<b>Top {TOP_TAGS} Etiquetas vs Top {TOP_TOPICOS} Tópicos</b>",
    xaxis_title="",
    yaxis_title="",
    height=max(600, TOP_TAGS * 32),
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="#fff",
    margin=dict(l=180, b=180)
)
fig_heatmap.update_xaxes(tickangle=-45)

fig_heatmap.write_html(OUTPUT_DIR / f"04_top{TOP_TAGS}_tags_vs_top{TOP_TOPICOS}_topicos_heatmap.html")
fig_heatmap.show()

Cruzemos las Notas vs la Cantidad de Tags Únicos. Y ademas calculemos un "Índice de Fragmentación" (cuántos tags se crean por cada nota) que sirva como una tercera dimensión (color): 

Calculamos el **Ratio de Fragmentación** ($\text{RF}$):

$$\text{RF} = \frac{\text{Tags Únicos Creados}}{\text{Notas con Tag del Tópico}}$$

**Interpretación:**

* **$\text{RF} \le \mathbf{0.20}$ Pocas etiquetas agrupan muchas notas (ej. 10 tags cubren 100 artículos).
* **$\mathbf{0.21} \le \text{RF} \le \mathbf{0.50}$:**  Etiquetas específicas o nombres propios.
* **$\text{RF} > \mathbf{0.50}$:**  Se inventan etiquetas para los articulos . Si roza o supera **1.0**, prácticamente se creó una nueva por cada nota publicada.

*Se filtran únicamente los tópicos con al menos 15 artículos para descartar distorsiones muestrales provocadas por temas marginales.*

In [26]:
# 1. Filtrar, calcular ratio y tomar los 15
df_ranking = (
    solapamiento[solapamiento["notas_con_tag"] >= 15]
    .assign(ratio_fragmentacion=lambda d: (d["tags_unicos_usados"] / d["notas_con_tag"]).round(2))
    .nlargest(15, "ratio_fragmentacion")
    .iloc[::-1]  # Invertir para que el mayor quede en la barra superior
)

# 2. Bar Chart Horizontal limpio
fig_ranking = px.bar(
    df_ranking,
    x="ratio_fragmentacion",
    y="topic_name",
    orientation="h",
    color="ratio_fragmentacion",
    color_continuous_scale="Reds",
    text="ratio_fragmentacion",
    hover_data={
        "topic": True,
        "notas_con_tag": True,
        "tags_unicos_usados": True,
        "ejemplos_tags": True,
    },
    title="<b>Tópicos con Mayor Dispersión de Etiquetas (Índice de Caos)</b><br><sup>Ratio = Tags únicos inventados / Total de notas del tema</sup>",
    labels={
        "ratio_fragmentacion": "Ratio de Caos",
        "topic_name": "Tópico Semántico",
    },
    height=600,
    template="plotly_white",
)

fig_ranking.update_traces(textposition="outside", texttemplate="%{text:.2f}")
fig_ranking.update_layout(
    xaxis_range=[0, df_ranking["ratio_fragmentacion"].max() * 1.18],
    coloraxis_showscale=False,
    margin=dict(l=180, r=40, t=80, b=40)
)

fig_ranking.write_html(OUTPUT_DIR / "05_ranking_caos_etiquetas.html")
fig_ranking.show()

## 4. Solapamiento de etiquetas



### Solapamiento por Coocurrencia en Documentos (Índice de Jaccard)

Este análisis es puramente cuantitativo, determinista y estructurado en dos fases:
* **Matriz Binaria Documento $\times$ Tag:** Cada artículo y cada etiqueta se cruzan en una tabla booleana donde $1$ indica presencia y $0$ ausencia.
* **Producto Punto Matricial (`.dot()`):** Se multiplica la matriz por su transpuesta para calcular de forma masiva cuántos artículos comparten simultáneamente cada par de etiquetas.

### Índice de Jaccard
Para evitar que una etiqueta gigantesca absorba a todas las demás por frecuencia, utilizamos el **Índice de Jaccard**, que mide la similitud y diversidad de conjuntos finitos. Su fórmula formal es:

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|} = \frac{\text{Articulos Compartidos por las etiquetas}}{\text{Total de articulos unicos entre ambas etiquetas}}$$

* **Intersección ($A \cap B$):** La cantidad exacta de artículos que llevan ambas etiquetas al mismo tiempo.
* **Unión ($A \cup B$):** El total de notas únicas abarcadas por la etiqueta A, la Tag B o por ambas simultáneamente (calculado como: $\text{Total}_A + \text{Total}_B - \text{Intersección}$).
* **Resultado:** Un coeficiente normalizado entre $0$ (cero solapamiento) y $1$ (idénticas).

### Parámetros de Corte y Filtro
* **Umbral de Jaccard ($> 0.15$):** Exige que al menos el 15% del universo combinado de notas esté efectivamente compartido entre ambas etiquetas.
* **Mínimo de Coocurrencia ($\ge 3$ notas)**

In [27]:
UMBRAL_JACCARD = 0.15          # Índice mínimo de solapamiento (0 a 1)
MIN_NOTAS_COMPARTIDAS = 3      # Cantidad mínima de artículos compartidos

# 1. Matriz binaria Documento x Tag 
df_binario = df_desanidado[["doc_id", "tag"]].drop_duplicates()
matriz_docs_tags = pd.crosstab(df_binario["doc_id"], df_binario["tag"])

# 2. Coocurrencia e Índice de Jaccard matricial
tags_cols = matriz_docs_tags.columns.to_numpy()
mat_val = matriz_docs_tags.to_numpy(dtype=np.int32)

# Multiplicación matricial: intersección A ∩ B
coocurrencia = mat_val.T @ mat_val

# Unión: Total_A + Total_B - Intersección
suma_tags = mat_val.sum(axis=0)
union = suma_tags[:, None] + suma_tags[None, :] - coocurrencia
jaccard = np.divide(coocurrencia, union, out=np.zeros_like(coocurrencia, dtype=float), where=union != 0)

# 3. Filtrado vectorizado del triángulo superior 
i_idx, j_idx = np.triu_indices(len(tags_cols), k=1)

mask = (jaccard[i_idx, j_idx] > UMBRAL_JACCARD) & (coocurrencia[i_idx, j_idx] >= MIN_NOTAS_COMPARTIDAS)

df_documentos = pd.DataFrame({
    "Tag A": tags_cols[i_idx[mask]],
    "Tag B": tags_cols[j_idx[mask]],
    "Notas_Compartidas": coocurrencia[i_idx[mask], j_idx[mask]],
    "Indice_Solapamiento_Jaccard": jaccard[i_idx[mask], j_idx[mask]].round(3)
}).sort_values(by="Indice_Solapamiento_Jaccard", ascending=False).reset_index(drop=True)

print(f"Pares con solapamiento significativo: {len(df_documentos):,}")
display(df_documentos.head(30))

# Exportar reporte sin pisar 'df'
df_documentos.to_csv(OUTPUT_DIR / "tags_solapamiento_documentos.csv", index=False)

Pares con solapamiento significativo: 107


,Tag A,Tag B,Notas_Compartidas,Indice_Solapamiento_Jaccard
0,Adolescencia,Salud Sexual Y Reproductiva,3,1.000
1,Adolescencia,Maternidad Temprana,3,1.000
2,Adolescencia,Anticoncepción,3,1.000
3,Anticoncepción,Maternidad Temprana,3,1.000
4,Anticoncepción,Salud Sexual Y Reproductiva,3,1.000
5,Maternidad Temprana,Salud Sexual Y Reproductiva,3,1.000
6,Francia Márquez,Gustavo Petro,3,0.750
7,Israel,Palestina,9,0.600
8,2001,Crisis 2001,10,0.588
9,Cop26,Crisis Climática,7,0.583


### Solapamiento semántico

El índice de Jaccard no puede diagnosticar etiquetas de frecuencia 1, se basa exclusivamente en la frecuencia.

Por lo tanto, utilizamos la **matriz de similitud semántica por *embeddings*** calculada  con `SentenceTransformer`. (utilizando las etiquetas no los textos de los articulos)

Con esto evaluamos el espacio vectorial de todo el vocabulario sin importar su frecuencia, el modelo de lenguaje detecta si un error tipográfico, un sinónimo menor o una variante etiqueta núcleo.


Se calcula la similitud coseno entre el conjunto de etiquetas unicas ($H$) y las etiquetas comunes ($C$, frecuencia $> 1$):

$$\text{Sim}(u, c) = \frac{\mathbf{e}_u \cdot \mathbf{e}_c}{\Vert{}\mathbf{e}_u\Vert{} \Vert{}\mathbf{e}_c\Vert{}} \quad \forall u \in U, \, c \in C$$

Para cada etiqueta unica se extrae su mas similar:

$$c^*(u) = \arg\max_{c \in C} \, \text{Sim}(u, c)$$
---

In [28]:
# 1. Catálogos sobre df_desanidado
tag_freqs = df_desanidado["tag"].value_counts()

tags_unicos_sospechosos = tag_freqs[tag_freqs == 1].index.tolist()
tags_nucleo = tag_freqs[tag_freqs > 1].index.tolist()

print(f"Total de tags únicos: {len(tag_freqs):,}")
print(f"Tags de uso único: {len(tags_unicos_sospechosos):,}")
print(f"Tags de núcleo (frecuencia > 1): {len(tags_nucleo):,}")

# 2. Generación directa de embeddings por conjunto
if 'embedding_model' not in globals():
    embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

print("Generando embeddings...")
emb_huerfanos = embedding_model.encode(tags_unicos_sospechosos, show_progress_bar=True)
emb_nucleo = embedding_model.encode(tags_nucleo, show_progress_bar=True)

# 3. Similitud Coseno
matriz_sim = cosine_similarity(emb_huerfanos, emb_nucleo)

# 4. Asignación vectorizada del match óptimo
mejores_idx = matriz_sim.argmax(axis=1)
mejores_scores = matriz_sim[np.arange(len(tags_unicos_sospechosos)), mejores_idx]

df_sugerencias = pd.DataFrame({
    "Etiqueta": tags_unicos_sospechosos,
    "Etiqueta Similar": np.array(tags_nucleo)[mejores_idx],
    "Similitud_Semantica": mejores_scores.round(3)
}).sort_values(by="Similitud_Semantica", ascending=False).reset_index(drop=True)

print(f"\nSe procesaron las sugerencias para los {len(df_sugerencias):,} tags huérfanos.")
display(df_sugerencias.head(40))

# Exportar reporte
df_sugerencias.to_csv(OUTPUT_DIR / "tags_huerfanos_sugerencias_fusion.csv", index=False)

Total de tags únicos: 881
Tags de uso único: 444
Tags de núcleo (frecuencia > 1): 437


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6738.63it/s]


Generando embeddings...


Batches: 100%|██████████| 14/14 [00:00<00:00, 16.35it/s]


Se procesaron las sugerencias para los 444 tags huérfanos.


,Etiqueta,Etiqueta Similar,Similitud_Semantica
0,Mexico,México,0.999
1,Iran,Irán,0.998
2,Crisiis Climática,Crisis Climática,0.993
3,Incendio,Incendios,0.989
4,Nicolás Maduro,Nicolas Maduro,0.988
5,Laboratorio Periodismo Performàtico,Laboratorio De Periodismo Performático,0.988
6,Policiriss,Policiales,0.985
7,Océano,Océanos,0.985
8,Culturas,Cultura,0.975
9,Crónicas,Crónica,0.964


Muchos tags aparecen en una sola nota o en muy pocas.

Implementamos un análisis basado en **distribuciones de probabilidad suave (*Soft Topic Distributions*)** mediante BERTopic.

Para cada articulo, realizar tres pasos:

1. **Distribución de Tópicos por Documento (`approximate_distribution`):** El modelo calcula un vector de probabilidad continua que indica qué tan alineada está una nota con *todos* los tópicos.
   
2. **Perfil Temático Agregado por Tag:** Cruzamos estas probabilidades con las etiquetas que posee la nota y calculamos el promedio de los vectores para cada tag.
   
3. **Similitud Coseno Vectorial:** Aplicamos producto escalar normalizado entre los topicos de las etiquetas. 

In [29]:
# 1. Distribución continua de tópicos sobre el df activo (sin releer de disco)
print("Calculando distribuciones de probabilidad con BERTopic...")
textos_docs = df["document_text_final"].fillna("").astype(str).tolist()
doc_topic_dist, _ = topic_model.approximate_distribution(textos_docs, use_embedding_model=False)

# 2. DataFrame de probabilidades alineado por doc_id real
df_dist = pd.DataFrame(doc_topic_dist, index=df["doc_id"])
col_topics = df_dist.columns.tolist()

# 3. Unir con df_desanidado (que ya está limpio y estandarizado)
df_tags_dist = df_desanidado[["doc_id", "tag"]].merge(df_dist, on="doc_id", how="inner")

# 4. Perfil temático promedio por tag
tag_topic_profiles = df_tags_dist.groupby("tag")[col_topics].mean()
tags_array = tag_topic_profiles.index.to_numpy()
matriz_perfiles = tag_topic_profiles.to_numpy()

# 5. Similitud Coseno matricial
print(f"Calculando similitud coseno para {len(tags_array):,} perfiles de tags...")
sim_matrix = cosine_similarity(matriz_perfiles)

# 6. Extracción vectorizada con NumPy 
UMBRAL_COSENO_PROB = 0.85
i_idx, j_idx = np.triu_indices(len(tags_array), k=1)
scores = sim_matrix[i_idx, j_idx]

# Máscara: supera umbral y descarta duplicados exactos (>= 0.999)
mask = (scores >= UMBRAL_COSENO_PROB) & (scores < 0.999)

df_pares_prob = pd.DataFrame({
    "Tag A": tags_array[i_idx[mask]],
    "Tag B": tags_array[j_idx[mask]],
    "Similitud_Coseno_Prob_Topicos": scores[mask].round(3)
}).sort_values(by="Similitud_Coseno_Prob_Topicos", ascending=False).reset_index(drop=True)

print(f"Pares con solapamiento temático alto: {len(df_pares_prob):,}")
display(df_pares_prob.head(40))

# Exportar reporte
df_pares_prob.to_csv(OUTPUT_DIR / "tags_solapamiento_prob_topicos.csv", index=False)

Calculando distribuciones de probabilidad con BERTopic...
Calculando similitud coseno para 881 perfiles de tags...
Pares con solapamiento temático alto: 14,081


,Tag A,Tag B,Similitud_Coseno_Prob_Topicos
0,Qatar 2022,Scaloneta,0.998
1,Messi,Qatar 2022,0.998
2,Di María,Messi,0.998
3,Qatar 2022,Selección Argentina,0.997
4,Iran,Medio Oriente,0.997
5,Fútbol,Mundial De Fútbol,0.996
6,Papa Francisco,Religion,0.996
7,Guerra,Ucrania,0.996
8,Di María,Scaloneta,0.996
9,Messi,Selección Argentina,0.996


El texto de Markdown para encabezar esta celda en el notebook queda estructurado de la siguiente manera:

---

### 2.4. Auditoría Léxica: Cuasi-duplicados y Variantes Tipográficas

Falto realizar una opcion mas simple, un control sintáctico detectar redundancias ortográficas y errores de tipeo.

#### Metodología de Detección

1. **Aislamiento Diacrítico:** Se descomponen los caracteres mediante normalización Unicode (`NFKD`) para remover tildes y diéresis. Si dos etiquetas coinciden exactamente tras este paso, se clasifican directamente como **Tilde / Diacrítico** (ej. `Iran` vs. `Irán`).
2. **Distancia Léxica:** Para el resto de los pares, se calcula el ratio de coincidencia de secuencias (*Gestalt Pattern Matching* vía `difflib`):

$$\text{Ratio} = \frac{2 \cdot M}{T}$$


Donde $M$ es la cantidad de caracteres coincidentes y $T$ es la suma de las longitudes de ambas cadenas.


3. **Poda por Longitud:** Se descartan de antemano pares con una diferencia mayor a 3 caracteres ($\vert{}len_A - len_B\vert{} > 3$), evitando comparaciones innecesarias entre cadenas incomparables.
4. **Criterio de Estandarización Canónica:** Cuando un par supera el umbral léxico ($\ge 0.82$), la forma con **mayor frecuencia histórica de uso** se asigna automáticamente como la versión estándar (*Tag Canónico*), mientras que la de menor uso se etiqueta como variante a absorber (*Tag Variante*).

---

In [30]:
# 1. Catálogo autocontenido directo desde df_desanidado
tag_counts = df_desanidado["tag"].value_counts()
todos_los_tags = tag_counts.index.tolist()

def normalizar_comparacion(texto):
    """Quita tildes y pasa a minúsculas para aislar variaciones de acentuación."""
    norm = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in norm if not unicodedata.combining(c)).lower().strip()

# Pre-computar textos normalizados y frecuencias en listas nativas (O(N) en lugar de O(N^2))
tags_norm = [normalizar_comparacion(t) for t in todos_los_tags]
tags_freq = tag_counts.values.tolist()

UMBRAL_SIMILITUD = 0.82
pares_similares = []
n = len(todos_los_tags)

# 2. Comparación rápida con poda por longitud
for i in range(n):
    tag_a, norm_a, freq_a = todos_los_tags[i], tags_norm[i], tags_freq[i]
    len_a = len(norm_a)

    for j in range(i + 1, n):
        norm_b = tags_norm[j]
        len_b = len(norm_b)

        # Poda: si difieren demasiado en caracteres, no pueden superar 0.82
        if abs(len_a - len_b) > 3:
            continue

        # Detección inmediata de tildes
        es_tilde = (norm_a == norm_b)
        ratio = 1.0 if es_tilde else difflib.SequenceMatcher(None, norm_a, norm_b).ratio()

        if ratio >= UMBRAL_SIMILITUD:
            tag_b, freq_b = todos_los_tags[j], tags_freq[j]

            # El de mayor frecuencia se propone como estándar editorial
            if freq_a >= freq_b:
                canonico, variante = tag_a, tag_b
                freq_canonico, freq_variante = freq_a, freq_b
            else:
                canonico, variante = tag_b, tag_a
                freq_canonico, freq_variante = freq_b, freq_a

            pares_similares.append({
                "Tag Variante": variante,
                "Frecuencia Variante": freq_variante,
                "Tag Sugerido (Canónico)": canonico,
                "Frecuencia Canónico": freq_canonico,
                "Similitud Léxica": round(float(ratio), 3),
                "Tipo de Desvío": "Tilde / Diacrítico" if es_tilde else "Plural / Tipeo"
            })

df_lexico = (
    pd.DataFrame(pares_similares)
    .sort_values(by=["Tipo de Desvío", "Similitud Léxica", "Frecuencia Variante"], ascending=[True, False, False])
    .reset_index(drop=True)
)

print(f"Total de pares casi-idénticos detectados: {len(df_lexico):,}")
display(df_lexico.head(40))

# Exportar reporte
df_lexico.to_csv(OUTPUT_DIR / "tags_cuasiduplicados_lexicos.csv", index=False)

Total de pares casi-idénticos detectados: 51


,Tag Variante,Frecuencia Variante,Tag Sugerido (Canónico),Frecuencia Canónico,Similitud Léxica,Tipo de Desvío
0,Crisiis Climática,1,Crisis Climática,11,0.970,Plural / Tipeo
1,Gabrielboric,2,Gabriel Boric,5,0.960,Plural / Tipeo
2,Adolescencias,2,Adolescencia,3,0.960,Plural / Tipeo
3,Laboratorio Periodismo Performàtico,1,Laboratorio De Periodismo Performático,2,0.959,Plural / Tipeo
4,Varones Gay,1,Varones Gays,1,0.957,Plural / Tipeo
5,Tecnologías,3,Tecnología,6,0.952,Plural / Tipeo
6,Violencia,7,Violencias,18,0.947,Plural / Tipeo
7,Femicidio,2,Femicidios,7,0.947,Plural / Tipeo
8,Incendio,1,Incendios,5,0.941,Plural / Tipeo
9,#Comunidadanfibia,1,Comunidad Anfibia,1,0.941,Plural / Tipeo


### Analisis Híbrido: Morfología, Lematización y Validación Semántica

Se implementa una arquitectura de filtrado en tres etapas:

#### 1. Lematización y Desambiguación Gramatical (spaCy)
Se extrae la raíz lingüística (*lema*) de cada etiqueta en minúsculas. Esto permite identificar de forma determinista variaciones de número (singular/plural como `Cuerpo` $\rightarrow$ `Cuerpos` o `Derecha` $\rightarrow$ `Derechas`) sin depender de heurísticas difusas de caracteres.

#### 2. Arbitraje Ortográfico
El pipeline aplica reglas normativas:
* **Prevalencia de diacríticos:** Entre variantes acentuadas y no acentuadas (`México` vs `Mexico`, `Nicolás Maduro` vs `Nicolas Maduro`), se impone la que lleva la tilde.
* **Depuración de sintaxis:** Las formas con numeral (`#Eaaf`) se subordinan a su contraparte textual limpia (`Eaaf`).

#### 3. Capa Semántica con Embeddings (`MiniLM-L12-v2`)
Para las variaciones léxicas que no constituyen lemas idénticos, se calcula la similitud coseno de sus representaciones vectoriales densas. Si dos términos presentan cercanía morfológica superficial pero divergencia conceptual ($\text{Sim}_{\text{sem}} < 0.70$), el par se descarta automáticamente, neutralizando falsos positivos como `Periodismo` / `Peronismo`.

#### 4. Salidas Operativas
* **`tags_fusiones_automaticas.csv`:** Casos de equivalencia técnica indudable (tildes, lemas validados y errores evidentes).
* **`tags_revision_manual.csv`:** Casos derivados a supervisión humana.

In [31]:
import spacy

# 1. Cargar modelos de NLP y Embeddings
if "nlp" not in globals():
    try:
        import es_core_news_md
        nlp = es_core_news_md.load(disable=["parser", "ner"])
    except ImportError:
        nlp = spacy.load("es_core_news_md", disable=["parser", "ner"])

if "embedding_model" not in globals():
    embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# 2. Catálogo autocontenido y frecuencias desde df_desanidado
tag_counts = df_desanidado["tag"].value_counts()
todos_los_tags = tag_counts.index.tolist()

def normalizar_comparacion(texto):
    """Remueve tildes, numerales y pasa a minúsculas."""
    texto_sin_hash = texto.lstrip("#").strip()
    norm = unicodedata.normalize("NFKD", texto_sin_hash)
    return "".join(c for c in norm if not unicodedata.combining(c)).lower().strip()

def extraer_lemas_lote(lista_textos, batch_size=256):
    """Lematiza forzando minúsculas para resolver flexiones nominales (singular/plural)."""
    textos_limpios = [t.lstrip("#").lower().strip() for t in lista_textos]
    lemas = []
    for doc in nlp.pipe(textos_limpios, batch_size=batch_size):
        lemas.append(" ".join([t.lemma_.lower() for t in doc if not t.is_punct and not t.is_space]))
    return lemas

# Pre-computar representaciones en memoria (O(N))
print("Pre-procesando catálogo, normalizando y extrayendo lemas...")
tags_norm = [normalizar_comparacion(t) for t in todos_los_tags]
tags_lemas = extraer_lemas_lote(todos_los_tags)
tags_freq = tag_counts.values.tolist()
n_tags = len(todos_los_tags)

# 3. Criba inicial morfológica y sintáctica
candidatos_lexicos = []

for i in range(n_tags):
    tag_a, norm_a, lema_a, freq_a = todos_los_tags[i], tags_norm[i], tags_lemas[i], tags_freq[i]
    len_a = len(norm_a)

    for j in range(i + 1, n_tags):
        tag_b, norm_b, lema_b, freq_b = todos_los_tags[j], tags_norm[j], tags_lemas[j], tags_freq[j]
        len_b = len(norm_b)

        # Si son diacríticos o hashtags idénticos
        es_diacritico = (norm_a == norm_b)
        
        # Mismo lema gramatical (singular vs plural)
        mismo_lema = (lema_a == lema_b) and (norm_a != norm_b) and bool(lema_a.strip())

        # Poda: si difieren demasiado en longitud y no comparten lema/diacrítico, saltar difflib
        if not (es_diacritico or mismo_lema) and abs(len_a - len_b) > 3:
            continue

        ratio_lexico = 1.0 if (es_diacritico or mismo_lema) else difflib.SequenceMatcher(None, norm_a, norm_b).ratio()

        if ratio_lexico >= 0.85 or es_diacritico or mismo_lema:
            
            # --- Criterio de Selección del Canónico ---
            # 1. Si uno es hashtag y el otro no, siempre gana el tag sin '#'
            es_hash_a = tag_a.startswith("#")
            es_hash_b = tag_b.startswith("#")
            
            if es_hash_a and not es_hash_b:
                canonico, variante = tag_b, tag_a
                freq_canonico, freq_variante = freq_b, freq_a
            elif es_hash_b and not es_hash_a:
                canonico, variante = tag_a, tag_b
                freq_canonico, freq_variante = freq_a, freq_b
            
            # 2. Si es diacrítico, gana la forma con tilde válida
            elif es_diacritico:
                tiene_tilde_a = any(c in "áéíóúÁÉÍÓÚ" for c in tag_a)
                tiene_tilde_b = any(c in "áéíóúÁÉÍÓÚ" for c in tag_b)
                if tiene_tilde_a and not tiene_tilde_b:
                    canonico, variante = tag_a, tag_b
                    freq_canonico, freq_variante = freq_a, freq_b
                elif tiene_tilde_b and not tiene_tilde_a:
                    canonico, variante = tag_b, tag_a
                    freq_canonico, freq_variante = freq_b, freq_a
                else:
                    # Si ambos tienen o no tildes, define frecuencia
                    if freq_a >= freq_b:
                        canonico, variante = tag_a, tag_b
                        freq_canonico, freq_variante = freq_a, freq_b
                    else:
                        canonico, variante = tag_b, tag_a
                        freq_canonico, freq_variante = freq_b, freq_a
            
            # 3. Resto de los casos: decide la frecuencia histórica
            else:
                if freq_a >= freq_b:
                    canonico, variante = tag_a, tag_b
                    freq_canonico, freq_variante = freq_a, freq_b
                else:
                    canonico, variante = tag_b, tag_a
                    freq_canonico, freq_variante = freq_b, freq_a

            candidatos_lexicos.append({
                "Tag Variante": variante,
                "Frecuencia Variante": int(freq_variante),
                "Tag Canónico": canonico,
                "Frecuencia Canónico": int(freq_canonico),
                "Similitud Léxica": round(float(ratio_lexico), 3),
                "Es Diacrítico": es_diacritico,
                "Mismo Lema": mismo_lema
            })

print(f"Pares candidatos pre-seleccionados: {len(candidatos_lexicos):,}")

# 4. Validación semántica vectorizada para casos no-deterministas
if candidatos_lexicos:
    df_candidatos = pd.DataFrame(candidatos_lexicos)

    # Identificar pares que no son diacríticos ni lemas exactos para validar significado
    df_necesitan_emb = df_candidatos[~(df_candidatos["Es Diacrítico"] | df_candidatos["Mismo Lema"])].copy()
    
    tags_a_validar = list(set(df_necesitan_emb["Tag Variante"]).union(set(df_necesitan_emb["Tag Canónico"])))
    print(f"Generando embeddings para {len(tags_a_validar):,} tags con posible ambigüedad...")

    if tags_a_validar:
        embeddings_dict = dict(zip(
            tags_a_validar, 
            embedding_model.encode(tags_a_validar, show_progress_bar=False)
        ))

        vec_var = np.vstack([embeddings_dict[t] for t in df_necesitan_emb["Tag Variante"]])
        vec_can = np.vstack([embeddings_dict[t] for t in df_necesitan_emb["Tag Canónico"]])

        sim_sem = np.sum(vec_var * vec_can, axis=1) / (
            np.linalg.norm(vec_var, axis=1) * np.linalg.norm(vec_can, axis=1)
        )
        df_necesitan_emb["Similitud Semántica"] = np.round(sim_sem, 3)
    else:
        df_necesitan_emb["Similitud Semántica"] = 1.0

    # Combinar métricas
    df_candidatos = df_candidatos.merge(
        df_necesitan_emb[["Tag Variante", "Tag Canónico", "Similitud Semántica"]],
        on=["Tag Variante", "Tag Canónico"],
        how="left"
    )
    df_candidatos["Similitud Semántica"] = df_candidatos["Similitud Semántica"].fillna(1.0)

    # 5. Filtrar falsos positivos semánticos (ej: Periodismo / Peronismo)
    mascara_validos = (
        df_candidatos["Es Diacrítico"] | 
        df_candidatos["Mismo Lema"] | 
        (df_candidatos["Similitud Semántica"] >= 0.70)
    )
    df_validados = df_candidatos[mascara_validos].copy()

    # 6. Segmentación operativa final
    condicion_automatica = (
        df_validados["Es Diacrítico"] |
        df_validados["Mismo Lema"] |
        ((df_validados["Similitud Léxica"] >= 0.93) & (df_validados["Similitud Semántica"] >= 0.85))
    )

    df_automaticas = (
        df_validados[condicion_automatica]
        .sort_values(by=["Es Diacrítico", "Mismo Lema", "Similitud Léxica"], ascending=[False, False, False])
        .reset_index(drop=True)
    )

    df_revision = (
        df_validados[~condicion_automatica]
        .sort_values(by=["Similitud Léxica", "Similitud Semántica"], ascending=[False, False])
        .reset_index(drop=True)
    )

    print(f"-> Fusiones automáticas seguras: {len(df_automaticas):,}")
    print(f"-> Casos para revisión editorial manual: {len(df_revision):,}")

    # Guardar reportes
    df_automaticas.to_csv(OUTPUT_DIR / "tags_fusiones_automaticas.csv", index=False)
    df_revision.to_csv(OUTPUT_DIR / "tags_revision_manual.csv", index=False)

    print("\n--- Vista Previa: Fusiones Automáticas ---")
    display(df_automaticas.head(30))

    print("\n--- Vista Previa: Revisión Editorial Manual ---")
    display(df_revision.head(30))
else:
    print("No se detectaron pares de tags para evaluar.")

Pre-procesando catálogo, normalizando y extrayendo lemas...
Pares candidatos pre-seleccionados: 43
Generando embeddings para 37 tags con posible ambigüedad...
-> Fusiones automáticas seguras: 26
-> Casos para revisión editorial manual: 5

--- Vista Previa: Fusiones Automáticas ---


,Tag Variante,Frecuencia Variante,Tag Canónico,Frecuencia Canónico,Similitud Léxica,Es Diacrítico,Mismo Lema,Similitud Semántica
0,Religion,6,Religión,8,1.000,True,False,1.000
1,Mexico,1,México,8,1.000,True,False,1.000
2,America Latina,2,América Latina,5,1.000,True,False,1.000
3,Iran,1,Irán,3,1.000,True,False,1.000
4,Nicolas Maduro,3,Nicolás Maduro,1,1.000,True,False,1.000
5,El Ultimo Rincon De Sudamerica,1,El Último Rincón De Sudamérica,3,1.000,True,False,1.000
6,Ley Omnibus,2,Ley Ómnibus,2,1.000,True,False,1.000
7,Politica Exterior,1,Política Exterior,1,1.000,True,False,1.000
8,#Eaaf,1,Eaaf,1,1.000,True,False,1.000
9,Derecha,2,Derechas,114,1.000,False,True,1.000



--- Vista Previa: Revisión Editorial Manual ---


,Tag Variante,Frecuencia Variante,Tag Canónico,Frecuencia Canónico,Similitud Léxica,Es Diacrítico,Mismo Lema,Similitud Semántica
0,Paternidades,1,Maternidades,7,0.917,False,False,0.918
1,Discursos Odio,1,Discursos De Odio,16,0.903,False,False,0.808
2,Arte,5,Artes,17,0.889,False,False,0.970
3,Neoliberalismo,6,Liberalismo,22,0.880,False,False,0.721
4,Madres De Plaza De Mayo,3,Abuelas De Plaza De Mayo,6,0.851,False,False,0.896


In [ ]:
# # ==============================================================================
# # EXPORTACIÓN PARA STREAMLIT (Módulo 3: Etiquetas y Auditorías)
# # 1. tags_vs_categorias.csv: Relación artículo-categoría-etiqueta con URL.
# # 2. tag_counts.csv: Frecuencias globales derivadas de la base anterior.
# # 3. tags_vs_topicos.parquet: Relación artículo-etiqueta-tópico.
# # 4. Reportes NLP (solapamiento temático, fusiones automáticas y manuales).
# # 5. Gráficos HTML pre-renderizados.
# # ==============================================================================

# import plotly.graph_objects as go
# from pathlib import Path
# import re
# import pandas as pd
# import shutil

# PROJECT_ROOT = Path.cwd().resolve()
# if PROJECT_ROOT.name == "notebooks":
#     PROJECT_ROOT = PROJECT_ROOT.parent

# STREAMLIT_DIR = PROJECT_ROOT / "data" / "streamlit"
# GRAFICOS_DIR = STREAMLIT_DIR / "graficos"
# STREAMLIT_DIR.mkdir(parents=True, exist_ok=True)
# GRAFICOS_DIR.mkdir(parents=True, exist_ok=True)

# # 1. Reconstruir tags_vs_categorias incluyendo doc_id, ID, title_text y source_url
# df_cat_exploded = (
#     df[["doc_id", "ID", "source_url", "title_text", "post_year", "category", "post_tag"]]
#     .copy()
#     .assign(
#         categoria_normalizada=lambda d: d["category"]
#         .fillna("Sin categoria")
#         .astype(str)
#         .apply(lambda x: [p.strip() for p in re.split(r"[;,]", x) if p.strip()] or ["Sin categoria"])
#     )
#     .explode("categoria_normalizada")
#     .reset_index(drop=True)
# )

# df_cruce = df_cat_exploded.dropna(subset=["post_tag"]).copy()
# df_cruce["tag"] = df_cruce["post_tag"].str.split(r"[;,]")
# df_cruce = df_cruce.explode("tag")
# df_cruce["tag"] = df_cruce["tag"].str.strip().str.title()
# df_cruce = df_cruce[df_cruce["tag"] != ""].drop_duplicates(subset=["doc_id", "tag", "categoria_normalizada"]).reset_index(drop=True)

# df_cruce.to_csv(STREAMLIT_DIR / "tags_vs_categorias.csv", index=False)

# # 2. Generar tag_counts (derivado de df_cruce para evitar huérfanos)
# tag_counts_global = (
#     df_cruce.drop_duplicates(subset=["doc_id", "tag"])["tag"]
#     .value_counts()
#     .reset_index()
# )
# tag_counts_global.columns = ["tag", "frecuencia"]
# tag_counts_global.to_csv(STREAMLIT_DIR / "tag_counts.csv", index=False)

# # 3. Exportar tags_vs_topicos.parquet
# cols_heatmap = ["tag", "topic_name", "doc_id"]
# # Mapeo directo de nombres oficiales ya consolidados
# df["topic_name"] = df["topic"].map(topic_info)
# # Asegurar que el 100 quede bautizado como Misceláneo
# df.loc[df["topic"] == 100, "topic_name"] = "100_outliers_miscelaneo"

# df_desanidado = (
#     df.dropna(subset=["post_tag"])
#     .copy()
#     .assign(tag=lambda d: d["post_tag"].str.split(r"[;,]"))
#     .explode("tag")
#     .assign(tag=lambda d: d["tag"].str.strip().str.title())
#     .loc[lambda d: (d["tag"] != "") & (d["tag"] != "Nan")]
#     .reset_index(drop=True)
# )
# df_tags_topics = df_desanidado[cols_heatmap].copy()
# df_tags_topics.to_parquet(STREAMLIT_DIR / "tags_vs_topicos.parquet", compression="snappy", index=False)

# # 4. Exportar reportes NLP de auditoría
# df_pares_prob.to_csv(STREAMLIT_DIR / "tags_solapamiento_prob_topicos.csv", index=False)
# df_automaticas.to_csv(STREAMLIT_DIR / "tags_fusiones_automaticas.csv", index=False)
# df_revision.to_csv(STREAMLIT_DIR / "tags_revision_manual.csv", index=False)

# # 5. Exportar HTMLs
# fig_pie = go.Figure(data=[go.Pie(labels=["Con Tags", "Sin Tags"], values=[con_tag, sin_tag], hole=.4)])
# fig_pie.update_layout(title="Cantidad de etiquetas en los articulos", width=500, height=400)
# fig_pie.write_html(GRAFICOS_DIR / "07_pie_cobertura.html")

# fig_evo.write_html(GRAFICOS_DIR / "07_evolucion_etiquetado.html")

# if 'fig_tags_cat' in globals():
#     fig_tags_cat.write_html(GRAFICOS_DIR / "07_distribucion_etiquetas_por_categoria.html")
# elif (OUTPUT_DIR / "03d_distribucion_etiquetas_por_categoria.html").exists():
#     shutil.copy2(OUTPUT_DIR / "03d_distribucion_etiquetas_por_categoria.html", GRAFICOS_DIR / "07_distribucion_etiquetas_por_categoria.html")

# print("Módulo de Etiquetas exportado correctamente y sin dependencias cruzadas.")

Módulo de Etiquetas exportado correctamente y sin dependencias cruzadas.
